<a href="https://colab.research.google.com/github/FaraahJ/Movie_Recommender_Project/blob/main/CollaborativeFiltering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/imdb_top_1000.csv')
df.drop(['Poster_Link', 'Released_Year', 'Certificate', 'Runtime', 'Genre', 'Overview', 'Meta_score','Director', 'Star1', 'Star2' , 'Star3', 'Star4', 'No_of_Votes', 'Gross'], axis=1, inplace=True)
df.head()

,Series_Title,IMDB_Rating
0,The Shawshank Redemption,9.3
1,The Godfather,9.2
2,The Dark Knight,9.0
3,The Godfather: Part II,9.0
4,12 Angry Men,9.0


In [4]:
!pip install plotly
from plotly.offline import init_notebook_mode, plot, iplot
import plotly.graph_objs as go
init_notebook_mode(connected=True)

In [5]:
data = df['IMDB_Rating'].value_counts().sort_index(ascending=False)
trace = go.Bar(x = data.index,
               text = ['{:.1f} %'.format(val) for val in (data.values / df.shape[0] * 100)],
               textposition = 'auto',
               textfont = dict(color = '#000000'),
               y = data.values,
               )

In [6]:
# Create layout
layout = dict(title = 'Distribution Of {} movie-ratings'.format(df.shape[0]),
              xaxis = dict(title = 'Rating'),
              yaxis = dict(title = 'Count'))
# Create plot
fig = go.Figure(data=[trace], layout=layout)
iplot(fig)

In [7]:
# Number of ratings per movie
data = df.groupby('Series_Title')['IMDB_Rating'].count().clip(upper=1000)

In [8]:
# Create trace
trace = go.Histogram(x = data.values,
                     name = 'Ratings',
                     xbins = dict(start = 0,
                                  end = 1000,
                                  size = 2))

In [9]:
# Create layout
layout = go.Layout(title = 'Distribution Of Number of Ratings Per Movie (Clipped at 1000)',
                   xaxis = dict(title = 'Number of Ratings Per Movie'),
                   yaxis = dict(title = 'Count'),
                   bargap = 0.2)

# Create plot
fig = go.Figure(data=[trace], layout=layout)
iplot(fig)

In [10]:
print('Data frame shape:\t{}'.format(df.shape))

Data frame shape:	(1000, 2)


In [33]:
min_movie_ratings = 0
filter_movies = df['Series_Title'].value_counts() > min_movie_ratings
filter_movies = filter_movies[filter_movies].index.tolist()
df_new = df[(df['Series_Title'].isin(filter_movies))]

In [34]:
#surprise library
!pip install surprise
!pip install similarities
!pip install faiss-cpu # Install faiss-cpu
import surprise as sp
import similarities as sims

In [35]:
from surprise import Reader, Dataset
# IMDB ratings are typically 0-10, so update the rating_scale
reader = Reader(rating_scale=(0, 10))
# Load data into surprise Dataset, specifying user_id, Series_Title (item_id), and IMDB_Rating
data = Dataset.load_from_df(df_new[['user_id', 'Series_Title', 'IMDB_Rating']], reader)

In [36]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/imdb_top_1000.csv')
df.drop(['Poster_Link', 'Released_Year', 'Certificate', 'Runtime', 'Genre', 'Overview', 'Meta_score','Director', 'Star1', 'Star2' , 'Star3', 'Star4', 'No_of_Votes', 'Gross'], axis=1, inplace=True)
# Add a user_id column for the surprise library (assuming a single user for this dataset)
df['user_id'] = 'user_01'
df.head()

,Series_Title,IMDB_Rating,user_id
0,The Shawshank Redemption,9.3,user_01
1,The Godfather,9.2,user_01
2,The Dark Knight,9.0,user_01
3,The Godfather: Part II,9.0,user_01
4,12 Angry Men,9.0,user_01


In [37]:
from surprise.model_selection import cross_validate, GridSearchCV
from surprise.prediction_algorithms import SVD, SVDpp, SlopeOne, NMF, NormalPredictor, KNNBasic, KNNBaseline, KNNWithMeans, BaselineOnly, CoClustering

In [39]:
from sklearn.metrics import root_mean_squared_error
from surprise.prediction_algorithms import SVD, SVDpp, SlopeOne, NMF, NormalPredictor, KNNBasic, KNNBaseline, KNNWithMeans, BaselineOnly, CoClustering
import pandas as pd # Ensure pandas is imported

benchmark = []
# Iterate over all algorithms
for algorithm in [SVD(), SVDpp(), SlopeOne(), NMF(), NormalPredictor(), KNNBaseline(), KNNBasic(), KNNWithMeans(), BaselineOnly(), CoClustering()]:
    # Perform cross validation
    results = cross_validate(algorithm, data, measures=['RMSE'], cv=3, verbose=False)

    # Get results & append algorithm name
    tmp = pd.DataFrame.from_dict(results).mean(axis=0)
    # Use pd.concat instead of the deprecated Series.append
    algorithm_series = pd.Series([str(algorithm).split(' ')[0].split('.')[-1]], index=['Algorithm'])
    tmp = pd.concat([tmp, algorithm_series])
    benchmark.append(tmp)

pd.DataFrame(benchmark).set_index('Algorithm').sort_values('test_rmse')

Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...


,test_rmse,fit_time,test_time
Algorithm,,,
KNNWithMeans,0.274890,0.000252,0.001362
KNNBasic,0.274961,0.000143,0.001387
CoClustering,0.275225,0.068429,0.001550
NMF,0.275262,0.042282,0.001540
KNNBaseline,0.275277,0.012146,0.001395
SVDpp,0.275284,0.725895,0.001652
SVD,0.275295,0.009837,0.001450
BaselineOnly,0.275489,0.001784,0.001225
SlopeOne,0.276246,0.025216,0.002047


In [40]:
#KNNWithMeans gives the best RMSE score

print('Using ALS')
knnwm_options = {'method': 'als',
               'n_epochs': 5,
               'reg_u': 12,
               'reg_i': 5
               }
algo = KNNWithMeans(knnwm_options=knnwm_options)
cross_validate(algo, data, measures=['RMSE'], cv=3, verbose=False)


Using ALS
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.


{'test_rmse': array([0.27958237, 0.26166231, 0.28459096]),
 'fit_time': (0.0003826618194580078,
  0.0003838539123535156,
  0.0003688335418701172),
 'test_time': (0.0032138824462890625,
  0.0016148090362548828,
  0.0014698505401611328)}

In [44]:
from surprise.model_selection import train_test_split
from surprise import accuracy

trainset, testset = train_test_split(data, test_size=0.25)
algo = KNNWithMeans(knnwm_options=knnwm_options)
predictions = algo.fit(trainset).test(testset)
accuracy.rmse(predictions)

Computing the msd similarity matrix...
Done computing similarity matrix.
RMSE: 0.2777


np.float64(0.277705640994201)

In [45]:
def get_Iu(uid):
    """ return the number of items rated by given user
    args:
      uid: the id of the user
    returns:
      the number of items rated by the user
    """
    try:
        return len(trainset.ur[trainset.to_inner_uid(uid)])
    except ValueError: # user was not part of the trainset
        return 0

def get_Ui(iid):
    """ return number of users that have rated given item
    args:
      iid: the raw id of the item
    returns:
      the number of users that have rated the item.
    """
    try:
        return len(trainset.ir[trainset.to_inner_iid(iid)])
    except ValueError:
        return 0

df = pd.DataFrame(predictions, columns=['uid', 'iid', 'rui', 'est', 'details'])
df['Iu'] = df.uid.apply(get_Iu)
df['Ui'] = df.iid.apply(get_Ui)
df['err'] = abs(df.est - df.rui)
best_predictions = df.sort_values(by='err')[:10]
worst_predictions = df.sort_values(by='err')[-10:]

In [46]:
best_predictions


,uid,iid,rui,est,details,Iu,Ui,err
15,user_01,Au revoir les enfants,8.0,7.953467,"{'was_impossible': True, 'reason': 'User and/o...",750,0,0.046533
10,user_01,Doctor Zhivago,8.0,7.953467,"{'was_impossible': True, 'reason': 'User and/o...",750,0,0.046533
39,user_01,The Killing,8.0,7.953467,"{'was_impossible': True, 'reason': 'User and/o...",750,0,0.046533
32,user_01,Touch of Evil,8.0,7.953467,"{'was_impossible': True, 'reason': 'User and/o...",750,0,0.046533
48,user_01,Baby,8.0,7.953467,"{'was_impossible': True, 'reason': 'User and/o...",750,0,0.046533
38,user_01,Rain Man,8.0,7.953467,"{'was_impossible': True, 'reason': 'User and/o...",750,0,0.046533
84,user_01,Bajrangi Bhaijaan,8.0,7.953467,"{'was_impossible': True, 'reason': 'User and/o...",750,0,0.046533
95,user_01,The Martian,8.0,7.953467,"{'was_impossible': True, 'reason': 'User and/o...",750,0,0.046533
96,user_01,A Streetcar Named Desire,8.0,7.953467,"{'was_impossible': True, 'reason': 'User and/o...",750,0,0.046533
100,user_01,Lion,8.0,7.953467,"{'was_impossible': True, 'reason': 'User and/o...",750,0,0.046533


In [47]:
worst_predictions

,uid,iid,rui,est,details,Iu,Ui,err
93,user_01,Whiplash,8.5,7.953467,"{'was_impossible': True, 'reason': 'User and/o...",750,0,0.546533
120,user_01,Léon,8.5,7.953467,"{'was_impossible': True, 'reason': 'User and/o...",750,0,0.546533
110,user_01,Modern Times,8.5,7.953467,"{'was_impossible': True, 'reason': 'User and/o...",750,0,0.546533
229,user_01,La vita è bella,8.6,7.953467,"{'was_impossible': True, 'reason': 'User and/o...",750,0,0.646533
236,user_01,Star Wars: Episode V - The Empire Strikes Back,8.7,7.953467,"{'was_impossible': True, 'reason': 'User and/o...",750,0,0.746533
150,user_01,The Lord of the Rings: The Two Towers,8.7,7.953467,"{'was_impossible': True, 'reason': 'User and/o...",750,0,0.746533
40,user_01,Fight Club,8.8,7.953467,"{'was_impossible': True, 'reason': 'User and/o...",750,0,0.846533
186,user_01,Forrest Gump,8.8,7.953467,"{'was_impossible': True, 'reason': 'User and/o...",750,0,0.846533
245,user_01,Inception,8.8,7.953467,"{'was_impossible': True, 'reason': 'User and/o...",750,0,0.846533
214,user_01,The Godfather,9.2,7.953467,"{'was_impossible': True, 'reason': 'User and/o...",750,0,1.246533


In [53]:
import matplotlib.pyplot as plt
%matplotlib notebook
df_new.loc[df_new['Series_Title'] == 'The Godfather']['IMDB_Rating'].hist()
plt.xlabel('rating')
plt.ylabel('Number of ratings')
plt.title('Number of ratings The Godfather movie has received')
plt.show();

In [54]:
#saving model
from surprise import dump

# 'algo' is your trained KNNWithMeans model
file_name = 'knn_with_means_model.pkl'

# Save the model state to disk
dump.dump(file_name, algo=algo)
print(f"Model successfully saved to {file_name}")

Model successfully saved to knn_with_means_model.pkl


In [ ]:
#loading model
from surprise import dump

file_name = 'knn_with_means_model.pkl'

# Load the saved model (the underscore skips the predictions tuple if empty)
_, loaded_algo = dump.load(file_name)

# Test it by predicting a rating for a user and item
prediction = loaded_algo.predict('user_id', 'IMDB_Rating', 'Series_title')
print(prediction.est)